In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform
import matplotlib.ticker as ticker

# 1. 파일 불러오기 (인코딩은 상황에 맞게 'euc-kr' 또는 'utf-8' 선택)
# 구글 시트에서 저장한 파일이라면 보통 'utf-8'입니다.
df = pd.read_csv('./data/real_estate_clean_v2.csv', encoding='cp949')
df

C:\Users\gorhk\AppData\Local\Temp\ipykernel_14612\2570270381.py:10: DtypeWarning: Columns (12,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/real_estate_clean_v2.csv', encoding='cp949')


,Unnamed: 0.1,Unnamed: 0,CGG_CD,CGG_NM,STDG_CD,STDG_NM,BLDG_NM,CTRT_DAY,THING_AMT,ARCH_AREA,...,RGHT_SE,ARCH_YR,BLDG_USG,DCLR_SE,CTRT_YEAR,QUARTER,QUARTER_INT,CTRT_BLDG_AGE,AMT_PER_AREA,AMT_PER_PYEONG
0,0,14,11530,구로구,10800,오류동,NaN,2018-12-31,73000,197.40,...,NaN,1990,단독다가구,NaN,2018,2018Q4,12,28,369.807497,1222.509625
1,1,16,11215,광진구,10100,중곡동,NaN,2018-12-31,102000,89.09,...,NaN,1973,단독다가구,NaN,2018,2018Q4,12,45,1144.909642,3784.842294
2,2,43,11560,영등포구,10900,영등포동8가,NaN,2018-12-29,88700,36.36,...,NaN,1945,단독다가구,NaN,2018,2018Q4,12,73,2439.493949,8064.479098
3,3,87,11230,동대문구,10900,휘경동,NaN,2018-12-28,95000,140.22,...,NaN,1991,단독다가구,NaN,2018,2018Q4,12,27,677.506775,2239.701897
4,4,92,11620,관악구,10200,신림동,NaN,2018-12-27,56500,179.46,...,NaN,1988,단독다가구,NaN,2018,2018Q4,12,30,314.833389,1040.776218
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
777302,777302,833566,11230,동대문구,11000,이문동,민족통일MJ캠퍼스외대4차,2023-12-04,21000,23.23,...,NaN,2023,오피스텔,중개거래,2023,2023Q4,12,0,904.003444,2988.454585
777303,777303,833567,11230,동대문구,11000,이문동,민족통일MJ캠퍼스외대4차,2023-12-04,21000,23.23,...,NaN,2023,오피스텔,중개거래,2023,2023Q4,12,0,904.003444,2988.454585
777304,777304,833568,11230,동대문구,11000,이문동,민족통일MJ캠퍼스외대4차,2023-12-04,21000,23.23,...,NaN,2023,오피스텔,중개거래,2023,2023Q4,12,0,904.003444,2988.454585
777305,777305,833569,11650,서초구,10800,서초동,벨라채 오피스텔,2023-12-04,24000,36.64,...,NaN,2003,오피스텔,중개거래,2023,2023Q4,12,20,655.021834,2165.371179


In [2]:
system_name = platform.system()
if system_name == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif system_name == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')

plt.rc('axes', unicode_minus=False)

In [4]:
df_ohryu = df[df['STDG_NM']=='오류동']
df_ohryu.shape

(4894, 22)

In [6]:
df_ohryu['BLDG_USG'].value_counts()

BLDG_USG
아파트      1951
연립다세대    1855
오피스텔      868
단독다가구     220
Name: count, dtype: int64

In [ ]:
## 자치구 별 기간 별 ROI, CAGR 계산기

# 1. 연도별, 지역별, 용도별 평균 평단가 계산
# 그룹핑 기준: 계약연도, 자치구명, 법정동명, 건물용도
# 이상치 제거를 위해 meidan을 사용할 수도 있지만, 요청하신 '평균'을 사용합니다.
price_trend = df.groupby(['CTRT_YEAR', 'CGG_NM', 'STDG_NM', 'BLDG_USG'])['AMT_PER_PYEONG'].mean().reset_index()

# 2. CAGR 계산 함수 정의
def calculate_cagr(gu, dong, bldg_type, start_year, duration):
    """
    특정 지역, 건물 유형, 투자 기간에 대한 CAGR(연평균 성장률)을 계산하는 함수
    
    Parameters:
    - gu: 자치구명 (예: '서대문구')
    - dong: 법정동명 (예: '대현동')
    - bldg_type: 건물용도 (예: '아파트')
    - start_year: 매수 연도 (예: 2018)
    - duration: 보유 기간 (년) (예: 6) -> 매도 연도 = start_year + duration
    """
    end_year = start_year + duration
    
    # 시작 연도 데이터 조회
    start_row = price_trend[
        (price_trend['CGG_NM'] == gu) &
        (price_trend['STDG_NM'] == dong) &
        (price_trend['BLDG_USG'] == bldg_type) &
        (price_trend['CTRT_YEAR'] == start_year)
    ]
    
    # 끝 연도 데이터 조회
    end_row = price_trend[
        (price_trend['CGG_NM'] == gu) &
        (price_trend['STDG_NM'] == dong) &
        (price_trend['BLDG_USG'] == bldg_type) &
        (price_trend['CTRT_YEAR'] == end_year)
    ]
    
    # 데이터 유효성 검사
    if start_row.empty:
        return f"Error: `{start_year}`년도 `{gu} {dong} {bldg_type}` 데이터가 없습니다."
    if end_row.empty:
        return f"Error: `{end_year}`년도 `{gu} {dong} {bldg_type}` 데이터가 없습니다."
    
    # 가격 추출 (Series 형태이므로 값만 추출)
    start_price = start_row['AMT_PER_PYEONG'].values[0]
    end_price = end_row['AMT_PER_PYEONG'].values[0]
    
    # CAGR 공식: (기말가치 / 기초가치)^(1/기간) - 1
    cagr = (end_price / start_price) ** (1 / duration) - 1
    
    # 결과 출력용 포맷팅
    print(f"[{gu} {dong} {bldg_type}]")
    print(f"- 매수({start_year}년): {start_price:,.0f} 만원/평")
    print(f"- 매도({end_year}년): {end_price:,.0f} 만원/평")
    print(f"- 총 수익률: {((end_price - start_price) / start_price) * 100:.2f}%")
    print(f"- CAGR (연평균 수익률): {cagr * 100:.2f}%")
    
    return cagr

# --- 사용 예시 ---
# 예: 서대문구 대현동 아파트를 2018년에 사서 6년(2024년) 뒤에 팔았다면?
result = calculate_cagr("서대문구", "대현동", "아파트", 2018, 6)

In [ ]:
def visualize_seoul_total_trend(bldg_type, start_year, start_quarter, duration_quarters):

    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 필터링 및 집계
    # 이미 존재하는 'QUARTER' 컬럼을 사용하여 필터링
    mask = (df['BLDG_USG'] == bldg_type) & (df['QUARTER'].isin(target_quarters))
    
    filtered_df = df.loc[mask]
    
    if filtered_df.empty:
        print("해당 조건의 데이터가 없습니다.")
        return

    # 분기별 평단가 평균 계산 (Index: QUARTER, Value: AMT_PER_PYEONG)
    # reindex를 사용하여 원하는 기간 순서대로 정렬 (누락된 기간은 NaN 처리 후 dropna)
    price_series = filtered_df.groupby('QUARTER')['AMT_PER_PYEONG'].mean()
    price_series = price_series.reindex(target_quarters).dropna()

    if price_series.empty:
        print("기간 내 유효한 데이터가 없습니다.")
        return

    # 3. 수익률 계산 (시작 시점 대비)
    start_price = price_series.iloc[0]
    return_series = (price_series / start_price - 1) * 100
    
    # 4. 시각화
    fig, ax1 = plt.subplots(figsize=(14, 7))
    
    # (1) 평단가 그래프 (좌측 축)
    color_price = 'tab:blue'
    # 제목에 시작 분기 표시
    ax1.set_title(f"서울시 전체 {bldg_type} 가격 및 누적 수익률 추이 ({target_quarters[0]} Base)", fontsize=16)
    ax1.set_xlabel('분기')
    ax1.set_ylabel('평당가 (만원/평)', color=color_price, fontsize=12)
    
    ax1.plot(price_series.index, price_series.values, color=color_price, marker='o', label='평균 평당가')
    ax1.tick_params(axis='y', labelcolor=color_price)
    ax1.grid(True, axis='x', linestyle='--', alpha=0.5)
    
    # Y축 천단위 콤마 포맷
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    # (2) 수익률 그래프 (우측 축)
    ax2 = ax1.twinx()
    color_ret = 'tab:red'
    ax2.set_ylabel('누적 수익률 (%)', color=color_ret, fontsize=12) 
    
    ax2.plot(return_series.index, return_series.values, color=color_ret, linestyle='--', marker='x', label='누적 수익률')
    ax2.tick_params(axis='y', labelcolor=color_ret)
    
    # 0% 기준선
    ax2.axhline(0, color='gray', linestyle=':', linewidth=1)
    
    # 마지막 시점 수익률 텍스트 표시
    last_idx = return_series.index[-1]
    last_val = return_series.iloc[-1]
    ax2.text(last_idx, last_val, f"{last_val:.1f}%", color='red', fontweight='bold', va='bottom', ha='left')
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # 5. 결과 요약 출력
    print(f"■ 서울시 전체 {bldg_type} 시장 흐름 요약")
    print(f"- 분석 기간: {len(price_series)}개 분기 ({price_series.index[0]} ~ {price_series.index[-1]})")
    print(f"- 시작 평당가: {start_price:,.0f} 만원")
    print(f"- 종료 평당가: {price_series.iloc[-1]:,.0f} 만원")
    print(f"- 서울시 전체 누적 수익률: {last_val:.2f}%")

# 실행 예시
visualize_seoul_total_trend("아파트", 2018, 1, 26)

In [ ]:
def visualize_mdd(bldg_type, start_year, start_quarter, duration_quarters, region='서울시'):
    """
    지역(서울시 전체 또는 특정 자치구)의 아파트 가격 MDD를 시각화합니다.
    df 전역 변수의 'QUARTER'(예: 2018Q1) 컬럼을 사용합니다.
    """
    
    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 필터링
    # 기본 마스크: 건물 용도 & 기간
    mask = (df['BLDG_USG'] == bldg_type) & (df['QUARTER'].isin(target_quarters))
    
    # [수정] 특정 자치구가 선택된 경우 조건 추가
    if region != '서울시':
        mask = mask & (df['CGG_NM'] == region)

    filtered_df = df.loc[mask]
    
    if filtered_df.empty:
        print(f"[{region}] 해당 기간에 데이터가 없습니다.")
        return

    # 분기별 평단가 평균 집계 (Index: QUARTER, Value: AMT_PER_PYEONG)
    # reindex를 사용하여 순서를 보장하고, 누락된 기간은 dropna로 처리
    trend_series = filtered_df.groupby('QUARTER')['AMT_PER_PYEONG'].mean()
    trend_series = trend_series.reindex(target_quarters).dropna()
    
    if trend_series.empty:
        print(f"[{region}] 선택한 기간에 유효한 데이터가 충분하지 않습니다.")
        return

    # Series -> DataFrame 변환
    trend_df = pd.DataFrame({'AMT_PER_PYEONG': trend_series})

    # 3. MDD 계산 (Run Max 대비 현재가 하락률)
    trend_df['Run_Max'] = trend_df['AMT_PER_PYEONG'].cummax()
    trend_df['Drawdown'] = (trend_df['AMT_PER_PYEONG'] / trend_df['Run_Max'] - 1) * 100
    
    mdd_val = trend_df['Drawdown'].min()
    mdd_point = trend_df['Drawdown'].idxmin()

    # 4. 시각화
    fig, ax = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    
    # (1) Upper Plot: 가격 및 전고점
    ax[0].plot(trend_df.index, trend_df['AMT_PER_PYEONG'], label='평균 평당가', color='blue', linewidth=2)
    ax[0].plot(trend_df.index, trend_df['Run_Max'], label='전고점 (Running Max)', color='green', linestyle='--', alpha=0.7)
    ax[0].fill_between(trend_df.index, trend_df['AMT_PER_PYEONG'], trend_df['Run_Max'], color='gray', alpha=0.1)
    
    ax[0].set_title(f"[{region}] {bldg_type} 가격 추이 및 전고점 비교", fontsize=16, fontweight='bold')
    ax[0].set_ylabel("평당가 (만원/평)")
    ax[0].legend(loc='upper left')
    ax[0].grid(True, linestyle='--', alpha=0.5)
    
    # Y축 천단위 콤마
    ax[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

    # (2) Lower Plot: Drawdown (Underwater Plot)
    ax[1].fill_between(trend_df.index, trend_df['Drawdown'], 0, color='red', alpha=0.3)
    ax[1].plot(trend_df.index, trend_df['Drawdown'], color='red', linewidth=1)
    
    # MDD 지점 표시
    ax[1].scatter(mdd_point, mdd_val, color='black', s=100, zorder=5)
    ax[1].text(mdd_point, mdd_val - 1, f"MDD: {mdd_val:.2f}%\n({mdd_point})", 
               ha='center', va='top', fontweight='bold', color='black')
    
    ax[1].set_title(f"Drawdown (낙폭) 추이 - 최대 낙폭(MDD): {mdd_val:.2f}%", fontsize=14)
    ax[1].set_ylabel("하락률 (%)")
    
    # Y축 범위 조정
    lower_lim = min(mdd_val * 1.5, -5) if mdd_val < 0 else -5
    ax[1].set_ylim(lower_lim, 1)
    
    ax[1].axhline(0, color='black', linewidth=1)
    ax[1].grid(True, linestyle='--', alpha=0.5)
    
    # X축 라벨 회전 (요청사항 반영)
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

    # 결과 요약
    print(f"■ [{region}] {bldg_type} MDD 분석 결과")
    print(f"- 최대 낙폭(MDD): {mdd_val:.2f}%")
    print(f"- MDD 발생 시점: {mdd_point} (전고점 {trend_df.loc[mdd_point, 'Run_Max']:,.0f}만원 대비)")
    print(f"- 현재 Drawdown: {trend_df['Drawdown'].iloc[-1]:.2f}%")

# --- 사용 예시 ---
visualize_mdd("아파트", 2018, 1, 27, region="종로구")

In [ ]:
def visualize_seoul_cagr_heatmap_ranked(bldg_type, start_year, start_quarter, duration_quarters):
    """
    자치구별 기간 내 누적 수익률 순위를 Heatmap으로 시각화합니다.
    """
    
    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 필터링
    mask = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type) &
        (df['CGG_NM'].notnull())
    )
    filtered_df = df.loc[mask].copy() 
    
    if filtered_df.empty:
        print("해당 조건의 데이터가 없습니다.")
        return

    # 3. [개별 자치구] 분기별 평단가 평균 집계 (Pivot)
    # Index: CGG_NM, Columns: QUARTER, Values: AMT_PER_PYEONG
    pivot_price = filtered_df.pivot_table(index='CGG_NM', columns='QUARTER', values='AMT_PER_PYEONG', aggfunc='mean')
    
    # 4. [서울시 전체] 분기별 평단가 평균 집계
    seoul_series = filtered_df.groupby('QUARTER')['AMT_PER_PYEONG'].mean()
    
    # 컬럼 순서 보장 (target_quarters 순서대로 정렬 및 reindex)
    # 데이터에 없는 분기가 있을 경우 NaN 처리됨
    valid_cols = [q for q in target_quarters if q in pivot_price.columns]
    
    # 누락된 기간이 너무 많으면 시각화가 어려우므로 경고
    if len(valid_cols) < 2:
        print("시각화할 기간 데이터가 충분하지 않습니다 (최소 2개 분기 필요).")
        return

    pivot_price = pivot_price[valid_cols]
    seoul_series = seoul_series.reindex(valid_cols)
    
    # 5. 수익률 계산 (기준 시점 대비 누적 수익률)
    # (1) 자치구 수익률
    base_prices = pivot_price.iloc[:, 0]  # 첫 번째 컬럼(시작 시점) 기준
    growth_rate_df = pivot_price.div(base_prices, axis=0) # 나눗셈
    growth_rate_df = (growth_rate_df - 1) * 100           # 백분율 변환
    
    # (2) 서울시 전체 수익률
    seoul_ret = (seoul_series / seoul_series.iloc[0] - 1) * 100
    
    # [핵심] '서울시 전체'를 로우(Row)로 추가
    growth_rate_df.loc['서울시 전체'] = seoul_ret
    
    # 6. 정렬 (마지막 시점 수익률 높은 순)
    final_returns = growth_rate_df.iloc[:, -1]
    final_sorted_df = growth_rate_df.loc[final_returns.sort_values(ascending=False).index]
    
    # 7. Heatmap 시각화
    plt.figure(figsize=(15, 12))
    
    sns.heatmap(final_sorted_df, annot=True, fmt='.1f', cmap='RdBu_r', center=0, linewidths=0.5)
    
    plt.title(f"서울시 자치구별 및 전체 {bldg_type} 누적 수익률 순위 ({valid_cols[0]} ~ {valid_cols[-1]})", fontsize=16)
    plt.xlabel("기간 (분기)")
    plt.ylabel("자치구 (수익률 상위 순)")
    
    # X축 라벨 45도 회전
    plt.xticks(rotation=45)
    
    # y축 라벨에서 '서울시 전체' 강조
    ax = plt.gca()
    for tick in ax.get_yticklabels():
        if tick.get_text() == '서울시 전체':
            tick.set_color('red')
            tick.set_fontweight('heavy')
            tick.set_fontsize(13)
        else:
            tick.set_color('black')
    
    plt.tight_layout()
    plt.show()
    
    # 순위 출력
    print(f"■ 최종 순위 요약 (총 {len(final_sorted_df)}개 지역)")
    rank = 1
    for name in final_sorted_df.index:
        ret = final_sorted_df.loc[name].iloc[-1]
        
        if name == '서울시 전체':
            print(f"★ [{rank}위] {name}: {ret:.2f}% (Average)")
        else:
            # 상위 3개, 하위 3개만 출력
            if rank <= 3 or rank >= len(final_sorted_df) - 2:
                print(f"{rank}위. {name}: {ret:.2f}%")
        rank += 1

# 실행 예시
visualize_seoul_cagr_heatmap_ranked("아파트", 2018, 1, 19)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import platform

# 폰트 및 마이너스 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
plt.rc('axes', unicode_minus=False)

def visualize_transaction_volume_vs_seoul(gu_name, start_year, start_quarter, duration_quarters, bldg_type="아파트"):
    """
    특정 자치구와 서울시 전체의 거래량을 설정된 기간 동안 연도별/분기별로 비교하여 시각화합니다.
    """
    
    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 준비 & 필터링
    # (1) 타겟 자치구
    mask_gu = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['CGG_NM'] == gu_name) & 
        (df['BLDG_USG'] == bldg_type)
    )
    gu_df = df.loc[mask_gu].copy()
    
    # (2) 서울시 전체 (비교군)
    mask_seoul = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type)
    )
    seoul_df = df.loc[mask_seoul].copy()

    if gu_df.empty:
        print(f"{gu_name} {bldg_type} 데이터가 없습니다.")
        return

    # 3. 연도별 거래량 집계
    gu_yearly = gu_df.groupby('CTRT_YEAR').size()
    seoul_yearly = seoul_df.groupby('CTRT_YEAR').size()

    # 4. 분기별 거래량 집계
    seoul_quarterly = seoul_df.groupby('QUARTER').size().sort_index()
    gu_quarterly = gu_df.groupby('QUARTER').size().sort_index()
    
    # 기간 맞추기 (Zero filling)
    seoul_quarterly = seoul_quarterly.reindex(target_quarters, fill_value=0)
    gu_quarterly = gu_quarterly.reindex(target_quarters, fill_value=0)
    
    # X축 라벨 생성 ('2019Q1' -> '19-1Q' 변환)
    q_labels = [f"{x[2:4]}-{x[5]}Q" for x in target_quarters]
    
    # 5. 시각화
    fig, ax = plt.subplots(2, 1, figsize=(14, 12))

    # --- (1) 연도별 Line Chart ---
    line_y1 = ax[0].plot(gu_yearly.index, gu_yearly.values, marker='o', color='royalblue', linewidth=3, label=f'{gu_name}')
    ax[0].set_ylabel(f"{gu_name} 거래 건수", color='royalblue', fontsize=12)
    ax[0].grid(axis='y', linestyle='--', alpha=0.5)
    
    # 오른쪽 축: 서울시 전체
    ax0_twin = ax[0].twinx()
    line_y2 = ax0_twin.plot(seoul_yearly.index, seoul_yearly.values, color='gray', marker='s', linestyle='--', linewidth=2, alpha=0.7, label='서울시 전체')
    ax0_twin.set_ylabel("서울시 전체 거래 건수", color='gray', fontsize=12)
    
    # X축 정수만 표시 및 라벨 추가
    ax[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax[0].set_title(f"{gu_name} vs 서울시 {bldg_type} 연도별 거래량 추이 ({target_quarters[0]} ~ {target_quarters[-1]})", fontsize=15)
    ax[0].set_xlabel("연도")  # [추가됨]
    
    # 범례 합치기
    lines_y = line_y1 + line_y2
    labels_y = [l.get_label() for l in lines_y]
    ax[0].legend(lines_y, labels_y, loc='upper left')

    # --- (2) 분기별 Line Chart ---
    x_range = range(len(q_labels))
    
    line_q1 = ax[1].plot(x_range, gu_quarterly.values, marker='o', color='coral', linewidth=2, label=f'{gu_name}')
    ax[1].set_ylabel(f"{gu_name} 거래 건수", color='coral', fontsize=12)
    ax[1].tick_params(axis='y', labelcolor='coral')
    
    # 오른쪽 축: 서울시 전체
    ax1_twin = ax[1].twinx()
    line_q2 = ax1_twin.plot(x_range, seoul_quarterly.values, marker='.', color='gray', linestyle='--', alpha=0.6, label='서울시 전체')
    ax1_twin.set_ylabel("서울시 전체 거래 건수", color='gray', fontsize=12)
    ax1_twin.tick_params(axis='y', labelcolor='gray')
    
    # X축 설정 및 라벨 추가
    ax[1].set_xticks(x_range)
    ax[1].set_xticklabels(q_labels, rotation=45) 
    ax[1].set_title(f"{gu_name} vs 서울시 {bldg_type} 분기별 거래량 흐름", fontsize=15)
    ax[1].set_xlabel("분기")  # [추가됨]
    ax[1].grid(True, linestyle='--', alpha=0.5)
    
    # 2022년 하락장 구간 강조
    indices_2022 = [i for i, label in enumerate(q_labels) if '22-' in label]
    if indices_2022:
        ax[1].axvspan(indices_2022[0]-0.5, indices_2022[-1]+0.5, color='red', alpha=0.1, label='2022년(하락장)')

    # 범례 표시
    lines_q = line_q1 + line_q2
    labels_q = [l.get_label() for l in lines_q]
    ax[1].legend(lines_q, labels_q, loc='upper left')

    plt.tight_layout()
    plt.show()

# 사용 예시
visualize_transaction_volume_vs_seoul("영등포구", 2018, 1, 19)

In [ ]:
# 1. 쿼리 조건 설정
target_year = 2022
target_quarters_nums = [1, 2, 3, 4] # 숫자 리스트
target_gu = ['종로구', '영등포구']

# QUARTER 컬럼 포맷(YYYYQx)에 맞춰 필터 리스트 생성
# 예: ['2022Q2', '2022Q3', '2022Q4']
target_quarters_str = [f"{target_year}Q{q}" for q in target_quarters_nums]

# 2. 데이터 필터링 (QUARTER 컬럼 사용)
condition = (
    (df['QUARTER'].isin(target_quarters_str)) &
    (df['CGG_NM'].isin(target_gu)) &
    (df['BLDG_USG'] == '아파트')
)

# 3. 결과 집계 (행: 자치구, 열: 분기)
# 이미 존재하는 QUARTER 컬럼으로 그룹핑
volume_result = df[condition].groupby(['CGG_NM', 'QUARTER']).size().unstack(fill_value=0)

print(f"=== {target_year}년 아파트 분기별 거래량 ({target_quarters_nums[0]}~{target_quarters_nums[-1]}분기) ===")
display(volume_result)

In [ ]:
# 1. 쿼리 조건 설정
target_year = 2022
target_quarters_nums = [1, 2, 3, 4] # 숫자 리스트
target_gu = ['종로구']

# QUARTER 컬럼 포맷(YYYYQx)에 맞춰 필터 리스트 생성
target_quarters_str = [f"{target_year}Q{q}" for q in target_quarters_nums]

# 2. 거래량 집계
# (1) 선택한 자치구의 거래량 구하기
condition_gu = (
    (df['QUARTER'].isin(target_quarters_str)) &
    (df['CGG_NM'].isin(target_gu)) &
    (df['BLDG_USG'] == '아파트')
)
volume_gu = df[condition_gu].groupby(['CGG_NM', 'QUARTER']).size().unstack(fill_value=0)

# (2) 서울시 전체 평균 거래량 구하기
condition_seoul = (
    (df['QUARTER'].isin(target_quarters_str)) &
    (df['BLDG_USG'] == '아파트')
)
# 분기별 서울시 전체 합계
seoul_total_vol = df[condition_seoul].groupby('QUARTER').size()

# 서울시 자치구 개수 (데이터에서 자동으로 확인, 보통 25개)
num_cgg = df['CGG_NM'].nunique()

# 평균 계산 (합계 / 자치구 수)
seoul_avg_vol = (seoul_total_vol / num_cgg).round(1)

# 3. 결과 합치기
# 기존 자치구 결과에 '서울시 평균' 행 추가
final_result = volume_gu.copy()
final_result.loc['서울시 평균'] = seoul_avg_vol

# 컬럼 순서가 뒤섞이지 않게 타겟 분기 순서대로 정렬 (옵션)
# final_result = final_result.reindex(columns=target_quarters_str)

print(f"=== {target_year}년 아파트 분기별 거래량 비교 (단위: 건) ===")
display(final_result)

In [ ]:
def visualize_seoul_qoq_heatmap_cum_sorted(bldg_type, start_year, start_quarter, duration_quarters):
    """
    자치구별 기간 내 QoQ(전분기 대비 증감률) 히트맵을 시각화합니다.
    정렬 기준은 기간 내 '누적 수익률'입니다.
    """
    
    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 필터링
    mask = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type) &
        (df['CGG_NM'].notnull())
    )
    filtered_df = df.loc[mask].copy()
    
    if filtered_df.empty:
        print("해당 조건의 데이터가 없습니다.")
        return

    # 3. 평단가 집계
    # (1) [개별 자치구] 분기별 평단가
    pivot_price = filtered_df.pivot_table(index='CGG_NM', columns='QUARTER', values='AMT_PER_PYEONG', aggfunc='mean')
    
    # (2) [서울시 전체] 분기별 평단가
    seoul_series = filtered_df.groupby('QUARTER')['AMT_PER_PYEONG'].mean()
    
    # 컬럼 순서 및 인덱스 맞추기 (누락된 분기 고려)
    valid_cols = [q for q in target_quarters if q in pivot_price.columns]
    
    if len(valid_cols) < 2:
        print("시각화할 기간 데이터가 충분하지 않습니다 (증감률 계산을 위해 최소 2개 분기 필요).")
        return

    pivot_price = pivot_price[valid_cols]
    seoul_series = seoul_series.reindex(valid_cols)
    
    # 4. QoQ (전분기 대비 증감률) 계산 -> 히트맵 색칠용
    # axis=1: 컬럼(분기) 간 계산
    qoq_df = pivot_price.pct_change(axis=1) * 100
    qoq_seoul = seoul_series.pct_change() * 100
    
    # 서울시 전체 행 추가
    qoq_df.loc['서울시 전체'] = qoq_seoul
    
    # 첫 분기는 증감률이 NaN이므로 제외
    qoq_df = qoq_df.iloc[:, 1:] 
    
    # 5. [핵심] 정렬 기준: 누적 수익률 (전체 기간)
    # (마지막 가격 / 시작 가격 - 1)
    
    # (1) 자치구 누적 수익률
    dist_cum_ret = (pivot_price.iloc[:, -1] / pivot_price.iloc[:, 0] - 1) * 100
    
    # (2) 서울시 전체 누적 수익률
    seoul_cum_ret = (seoul_series.iloc[-1] / seoul_series.iloc[0] - 1) * 100
    
    # 합쳐서 정렬용 Series 생성
    dist_cum_ret['서울시 전체'] = seoul_cum_ret
    
    # 누적 수익률 높은 순서로 정렬된 인덱스 추출
    sorted_indices = dist_cum_ret.sort_values(ascending=False).index
    
    # QoQ 데이터프레임을 이 순서대로 재정렬
    final_sorted_df = qoq_df.loc[sorted_indices]
    
    if final_sorted_df.empty:
        print("비교할 기간이 충분하지 않습니다.")
        return

    # 6. Heatmap 시각화
    plt.figure(figsize=(15, 12))
    
    sns.heatmap(final_sorted_df, annot=True, fmt='.1f', cmap='RdBu_r', center=0, linewidths=0.5, vmin=-5, vmax=5)
    
    plt.title(f"서울시 자치구별 QoQ 증감률 히트맵 (정렬: 누적 수익률 순)", fontsize=16)
    plt.xlabel("기간 (분기)")
    plt.ylabel("자치구 (누적 수익률 High -> Low)")
    
    # X축 라벨 회전 (45도)
    plt.xticks(rotation=45)
    
    # '서울시 전체' 라벨 강조 (Y축)
    ax = plt.gca()
    for tick in ax.get_yticklabels():
        if tick.get_text() == '서울시 전체':
            tick.set_color('red')
            tick.set_fontweight('heavy')
            tick.set_fontsize(13)
        else:
            tick.set_color('black')
            
    plt.tight_layout()
    plt.show()
    
    # 상위 1등과 서울 평균 출력
    # (주의: sorted_indices에는 서울시 전체도 포함되어 있으므로 구분 필요)
    top_1 = sorted_indices[0] if sorted_indices[0] != '서울시 전체' else sorted_indices[1]
    top_val = dist_cum_ret[top_1]
    
    print(f"■ 누적 수익률 1위 자치구: {top_1} ({top_val:.2f}%)")
    print(f"■ 서울시 전체 누적 수익률: {seoul_cum_ret:.2f}%")

# 실행 예시
visualize_seoul_qoq_heatmap_cum_sorted("아파트", 2018, 1, 19)

In [ ]:
import pandas as pd
import platform
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def run_real_estate_backtest_vs_seoul(target_gu, bldg_type, start_year, start_quarter, initial_capital, duration_quarters):
    """
    특정 자치구와 서울시 전체의 부동산 투자 백테스트를 수행하고 비교합니다. (CAGR/MDD 제거)
    """
    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 필터링
    # (1) 타겟 자치구
    mask_gu = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type) &
        (df['CGG_NM'] == target_gu)
    )
    df_gu = df.loc[mask_gu].copy()
    
    # (2) 서울시 전체 (비교군)
    mask_seoul = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type)
    )
    df_seoul = df.loc[mask_seoul].copy()
    
    if df_gu.empty or df_seoul.empty:
        print("데이터가 부족합니다.")
        return

    # 공통 전처리 함수 (분기별 평단가 Series 생성)
    def get_price_series(temp_df):
        # Index: QUARTER, Value: AMT_PER_PYEONG
        series = temp_df.groupby('QUARTER')['AMT_PER_PYEONG'].mean()
        series = series.reindex(target_quarters).dropna()
        return series

    price_gu = get_price_series(df_gu)
    price_seoul = get_price_series(df_seoul)
    
    # 데이터가 중간에 끊겼으면 교집합 기간만 비교
    common_idx = price_gu.index.intersection(price_seoul.index)
    if common_idx.empty:
        print("겹치는 기간의 데이터가 없습니다.")
        return
        
    price_gu = price_gu.loc[common_idx]
    price_seoul = price_seoul.loc[common_idx]
    
    # 3. 자산 가치 변동 및 지표 계산 함수 (CAGR, MDD 제거)
    def calc_metrics(price_series, capital):
        start_p = price_series.iloc[0]
        # (현재가 / 시작가) * 원금 = 현재 평가금액
        portfolio = (price_series / start_p) * capital
        
        final_val = portfolio.iloc[-1]
        ret = (final_val / capital - 1) * 100
        
        return portfolio, final_val, ret

    # 계산 수행
    pf_gu, val_gu, ret_gu = calc_metrics(price_gu, initial_capital)
    pf_seoul, val_seoul, ret_seoul = calc_metrics(price_seoul, initial_capital)
    
    profit_gu = val_gu - initial_capital
    profit_seoul = val_seoul - initial_capital
    
    # --- [결과 텍스트 출력] (단위: 만원) ---
    print(f"================================================================")
    print(f"      [{target_gu} vs 서울시 전체] {bldg_type} 투자 백테스트 비교")
    print(f"================================================================")
    print(f"1. 투자 기간 : {common_idx[0]} ~ {common_idx[-1]} ({len(common_idx)}분기)")
    print(f"2. 매입 금액 : {initial_capital/10000:,.0f}만 원")
    print(f"----------------------------------------------------------------")
    print(f"   구분        |      {target_gu}      |     서울시 전체")
    print(f"----------------------------------------------------------------")
    print(f"3. 최종 금액   |  {val_gu/10000:,.0f}만 ({profit_gu/10000:+,.0f}만) |  {val_seoul/10000:,.0f}만 ({profit_seoul/10000:+,.0f}만)")
    print(f"4. 누적 수익률 |     {ret_gu:6.2f} %       |     {ret_seoul:6.2f} %")
    print(f"----------------------------------------------------------------")
    print(f" * 비교 결과: {target_gu} 수익률이 서울 평균보다 {'높음' if ret_gu > ret_seoul else '낮음'}")
    print(f"================================================================\n")
    
    # --- [시각화] ---
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # (1) 데이터 플로팅
    ax.plot(pf_gu.index, pf_gu, color='red', linewidth=2.5, marker='o', label=f'{target_gu}')
    ax.plot(pf_seoul.index, pf_seoul, color='gray', linewidth=2, linestyle='--', marker='s', label='서울시 전체')
    
    # (2) 기준선 (원금) 강조
    ax.axhline(initial_capital, color='black', linewidth=1.5, linestyle='-', zorder=1)
    ax.text(pf_gu.index[0], initial_capital, " 원금 (Start)", va='bottom', fontweight='bold')
    
    # (3) 영역 채우기
    ax.fill_between(pf_gu.index, pf_gu, initial_capital, where=(pf_gu >= initial_capital), color='red', alpha=0.1)
    ax.fill_between(pf_gu.index, pf_gu, initial_capital, where=(pf_gu < initial_capital), color='blue', alpha=0.1)
    
    ax.set_title(f"10억 투자 시 자산 가치 변화 비교 ({target_gu} vs 서울 평균)", fontsize=15)
    ax.set_ylabel("평가 금액 (만원)")
    
    # Y축 포맷 (만원 단위)
    def ten_thousands(x, pos):
        return f'{x/10000:,.0f}만'
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(ten_thousands))
    
    # 최종 금액 라벨링 (만원 단위)
    ax.text(pf_gu.index[-1], pf_gu.iloc[-1], f"{pf_gu.iloc[-1]/10000:,.0f}만", color='red', fontweight='bold', ha='left', va='center')
    ax.text(pf_seoul.index[-1], pf_seoul.iloc[-1], f"{pf_seoul.iloc[-1]/10000:,.0f}만", color='gray', fontweight='bold', ha='left', va='center')

    # X축 라벨 회전 (45도)
    plt.xticks(rotation=45)
    
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

# 실행 예시
run_real_estate_backtest_vs_seoul("종로구", "아파트", 2022, 1, 1000000000, 11)

In [ ]:
import pandas as pd
import platform
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def run_real_estate_backtest_inter_gu(gu1, gu2, bldg_type, start_year, start_quarter, initial_capital, duration_quarters):
    """
    [자치구 vs 자치구] 두 지역의 부동산 투자 백테스트를 수행하고 비교합니다. (CAGR/MDD 제거)
    """
    # 0. 건물 용도 데이터 존재 여부 검증
    # df는 전역 변수로 가정
    available_types = df['BLDG_USG'].unique()
    if bldg_type not in available_types:
        print(f"⚠️ 오류: 요청하신 건물 용도 '{bldg_type}'는 데이터셋에 존재하지 않습니다.")
        print(f"   (사용 가능한 용도: {list(available_types)})")
        return

    # 1. 분석 대상 기간 리스트 생성 (포맷: 2018Q1)
    target_quarters = []
    current_y, current_q = start_year, start_quarter
    
    for _ in range(duration_quarters + 1):
        target_quarters.append(f"{current_y}Q{current_q}")
        current_q += 1
        if current_q > 4:
            current_y += 1
            current_q = 1
            
    # 2. 데이터 필터링
    # (1) 첫 번째 자치구 (gu1)
    mask_1 = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type) &
        (df['CGG_NM'] == gu1)
    )
    df_1 = df.loc[mask_1].copy()
    
    # (2) 두 번째 자치구 (gu2)
    mask_2 = (
        (df['QUARTER'].isin(target_quarters)) & 
        (df['BLDG_USG'] == bldg_type) &
        (df['CGG_NM'] == gu2)
    )
    df_2 = df.loc[mask_2].copy()
    
    if df_1.empty:
        print(f"[{gu1}] 해당 기간/용도의 데이터가 부족합니다.")
        return
    if df_2.empty:
        print(f"[{gu2}] 해당 기간/용도의 데이터가 부족합니다.")
        return

    # 공통 전처리 함수 (분기별 평단가 Series 생성)
    def get_price_series(temp_df):
        # Index: QUARTER, Value: AMT_PER_PYEONG
        series = temp_df.groupby('QUARTER')['AMT_PER_PYEONG'].mean()
        series = series.reindex(target_quarters).dropna()
        return series

    price_1 = get_price_series(df_1)
    price_2 = get_price_series(df_2)
    
    # 데이터가 중간에 끊겼으면 교집합 기간만 비교
    common_idx = price_1.index.intersection(price_2.index)
    if common_idx.empty:
        print("두 지역 간 겹치는 거래 기간이 없습니다.")
        return
        
    price_1 = price_1.loc[common_idx]
    price_2 = price_2.loc[common_idx]
    
    # 3. 자산 가치 변동 및 수익률 계산 함수 (CAGR, MDD 제거)
    def calc_metrics(price_series, capital):
        start_p = price_series.iloc[0]
        # (현재가 / 시작가) * 원금 = 현재 평가금액
        portfolio = (price_series / start_p) * capital
        
        final_val = portfolio.iloc[-1]
        ret = (final_val / capital - 1) * 100
        
        return portfolio, final_val, ret

    # 계산 수행
    pf_1, val_1, ret_1 = calc_metrics(price_1, initial_capital)
    pf_2, val_2, ret_2 = calc_metrics(price_2, initial_capital)
    
    profit_1 = val_1 - initial_capital
    profit_2 = val_2 - initial_capital
    
    # --- [결과 텍스트 출력] (단위: 만원) ---
    print(f"================================================================")
    print(f"      [{gu1} vs {gu2}] {bldg_type} 투자 백테스트 비교")
    print(f"================================================================")
    print(f"1. 투자 기간 : {common_idx[0]} ~ {common_idx[-1]} ({len(common_idx)}분기)")
    print(f"2. 매입 금액 : {initial_capital/10000:,.0f}만 원")
    print(f"----------------------------------------------------------------")
    print(f"   구분        |      {gu1}      |     {gu2}")
    print(f"----------------------------------------------------------------")
    print(f"3. 최종 금액   |  {val_1/10000:,.0f}만 ({profit_1/10000:+,.0f}만) |  {val_2/10000:,.0f}만 ({profit_2/10000:+,.0f}만)")
    print(f"4. 누적 수익률 |     {ret_1:6.2f} %       |     {ret_2:6.2f} %")
    print(f"----------------------------------------------------------------")
    
    better_gu = gu1 if ret_1 > ret_2 else gu2
    print(f" * 비교 결과: {better_gu}의 수익률이 더 높습니다.")
    print(f"================================================================\n")
    
    # --- [시각화] ---
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # (1) 데이터 플로팅
    # gu1: 빨간색 실선, gu2: 파란색 실선 (서울시는 보통 회색 점선으로 표현하지만 대등한 비교이므로 실선 사용)
    ax.plot(pf_1.index, pf_1, color='#FF5A5A', linewidth=2.5, marker='o', label=f'{gu1}')
    ax.plot(pf_2.index, pf_2, color='#5A5AFF', linewidth=2.5, marker='s', label=f'{gu2}')
    
    # (2) 기준선 (원금) 강조
    ax.axhline(initial_capital, color='black', linewidth=1.5, linestyle='-', zorder=1)
    ax.text(pf_1.index[0], initial_capital, " 원금 (Start)", va='bottom', fontweight='bold')
    
    # (3) 영역 채우기 (수익률 더 높은 쪽 강조 - 복잡해질 수 있어 단순 상승/하락보다는 두 선 사이 영역 채우기 고려 가능하나, 여기선 생략하거나 간단히 처리)
    # 여기서는 각 선 아래를 연하게 채워 시각적 볼륨감 부여
    ax.fill_between(pf_1.index, pf_1, initial_capital, color='#FF5A5A', alpha=0.05)
    ax.fill_between(pf_2.index, pf_2, initial_capital, color='#5A5AFF', alpha=0.05)
    
    ax.set_title(f"10억 투자 시 자산 가치 변화 비교 ({gu1} vs {gu2})", fontsize=15)
    ax.set_ylabel("평가 금액 (만원)")
    
    # Y축 포맷 (만원 단위)
    def ten_thousands(x, pos):
        return f'{x/10000:,.0f}만'
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(ten_thousands))
    
    # 최종 금액 라벨링
    ax.text(pf_1.index[-1], pf_1.iloc[-1], f"{pf_1.iloc[-1]/10000:,.0f}만", color='#FF5A5A', fontweight='bold', ha='left', va='center')
    ax.text(pf_2.index[-1], pf_2.iloc[-1], f"{pf_2.iloc[-1]/10000:,.0f}만", color='#5A5AFF', fontweight='bold', ha='left', va='center')

    # X축 라벨 회전 (45도)
    plt.xticks(rotation=45)
    
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

# 실행 예시: 종로구 vs 강남구, 아파트, 2018년 1분기부터 20분기(5년) 동안
run_real_estate_backtest_inter_gu("중랑구", "용산구", "오피스텔", 2018, 1, 300000000, 27)

In [ ]:
# 1. 건물 용도별, 분기별 평균 평당가 집계
# 서울시 전체 기준이므로 자치구(CGG_NM) 구분 없이 집계
# df 전역 변수의 'QUARTER' 컬럼(예: 2018Q1) 사용
pivot_df = df.groupby(['BLDG_USG', 'QUARTER'])['AMT_PER_PYEONG'].mean().unstack(level=0)

# 시계열 순서가 섞이지 않게 인덱스 정렬
trend_df = pivot_df.sort_index()

# 널 값이 있는 경우(특정 분기에 거래가 없는 경우) 보간하거나 drop해야 함
# 여기서는 시각화를 끊기지 않게 하기 위해 앞의 값으로 채움 (선택 사항)
trend_df = trend_df.fillna(method='ffill')

# 2. 누적 수익률 계산 (Cumulative Return)
# (현재가 - 시작가) / 시작가 * 100
# 시작점(첫 데이터)을 기준(0%)으로 잡습니다.
cumulative_return = trend_df.apply(lambda x: (x / x.iloc[0] - 1) * 100)

# 3. 시각화
plt.figure(figsize=(12, 7))

# X축 라벨 생성 (2018Q1 -> 18-1Q 변환)
x_labels = [f"{x[2:4]}-{x[5]}Q" for x in cumulative_return.index]

# 각 건물 용도별로 선 그리기
x_range = range(len(x_labels))

for col in cumulative_return.columns:
    # 아파트는 중요하니 굵고 진하게 강조
    if col == '아파트':
        plt.plot(x_range, cumulative_return[col], label=col, linewidth=3.5, marker='o', zorder=10)
    else:
        plt.plot(x_range, cumulative_return[col], label=col, linewidth=1.5, linestyle='--', alpha=0.7)

# 4. 그래프 꾸미기
plt.title('서울시 건물 용도별 분기 누적 수익률 추이 (시작 시점=0%)', fontsize=15)
plt.ylabel('누적 수익률 (%)')
plt.xlabel('기간 (분기)')

# X축 라벨 설정 (너무 빽빽하지_않게 2분기 간격으로 표시 예시)
# 전체를 다 보여주고 싶으면 slice(::2) 제거
plt.xticks(x_range[::2], x_labels[::2], rotation=45)

# 기준선(0%) 표시
plt.axhline(0, color='black', linewidth=1, linestyle='-')
plt.grid(True, linestyle='--', alpha=0.5)

# Y축 % 포맷
plt.gca().yaxis.set_major_formatter(ticker.PercentFormatter(decimals=0))

plt.legend()
plt.tight_layout()
plt.show()

# 최종 수익률 출력
print("=== 기간 내 최종 누적 수익률 ===")
print(cumulative_return.iloc[-1].sort_values(ascending=False).apply(lambda x: f"{x:+.2f}%"))